In [1]:
!pip -q install datasets

Literature

In [2]:
from huggingface_hub import login

login("")

In [3]:
from datasets import load_dataset

ds = load_dataset("Vacaspati/Vacaspati")

print(ds)

print("\nColumns:")
print(ds["train"].column_names)

print("\nSample:")
print(ds["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 4667936
    })
})

Columns:
['text']

Sample:
{'text': 'সাময়িক পত্রে প্রকাশিত'}


In [4]:
import unicodedata
import re

count = 0

with open("bn_literature.txt", "w", encoding="utf-8") as out:

    for row in ds["train"]:

        text = row["text"]

        text = unicodedata.normalize("NFC", text)
        text = re.sub(r"\s+", " ", text).strip()

        if not text:
            continue

        out.write(text + "\n")
        count += 1

print("Saved documents:", count)

Saved documents: 4604919


In [5]:
import random

TARGET_WORDS = 5601171

with open("bn_literature.txt", encoding="utf-8") as f:
    docs = [x.strip() for x in f if x.strip()]

random.seed(42)
random.shuffle(docs)

selected = []
word_count = 0

for doc in docs:
    wc = len(doc.split())

    if word_count + wc > TARGET_WORDS:
        continue

    selected.append(doc)
    word_count += wc

    if word_count >= TARGET_WORDS:
        break

print("Selected documents:", len(selected))
print("Selected words:", word_count)

with open(
    "literature_balanced.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write("\n".join(selected))

Selected documents: 251038
Selected words: 5601171


In [6]:
from collections import Counter

file_path = "literature_balanced.txt"

docs = 0
words = 0

with open(file_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            docs += 1
            words += len(line.split())

print("Documents:", docs)
print("Words:", words)

Documents: 251038
Words: 5601171


In [7]:
from collections import Counter

file_path = "literature_balanced.txt"

docs = 0
words = 0
chars = 0
vocab = Counter()

with open(file_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        docs += 1
        chars += len(line)

        ws = line.split()
        words += len(ws)

        vocab.update(ws)

print("Documents :", docs)
print("Characters:", chars)
print("Words     :", words)
print("Vocabulary:", len(vocab))
print("Avg Words/Doc:", round(words/docs, 2))

Documents : 251038
Characters: 33938296
Words     : 5601171
Vocabulary: 519345
Avg Words/Doc: 22.31


News

In [8]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "ai4bharat/IndicCorpV2",
    repo_type="dataset"
)

print(len(files))
print(files[:20])

28
['.gitattributes', 'README.md', 'data/as.txt', 'data/bd.txt', 'data/bn.txt', 'data/dg.txt', 'data/en.txt', 'data/gom.txt', 'data/gu.txt', 'data/hi-1.txt', 'data/hi-2.txt', 'data/hi-3.txt', 'data/kha.txt', 'data/kn.txt', 'data/ks.txt', 'data/mai.txt', 'data/ml.txt', 'data/mni.txt', 'data/mr.txt', 'data/ne.txt']


In [9]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    data_files={"train": "data/bn.txt"}
)

print(dataset)

Loading dataset shards:   0%|          | 0/33 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 41004792
    })
})


In [10]:
import unicodedata
import re

MAX_DOCS = 500000

with open("bn_news.txt", "w", encoding="utf-8") as f:

    for i, row in enumerate(dataset["train"]):

        text = unicodedata.normalize("NFC", row["text"])
        text = re.sub(r"\s+", " ", text).strip()

        if text:
            f.write(text + "\n")

        if i + 1 >= MAX_DOCS:
            break

print("Saved", MAX_DOCS, "documents")

Saved 500000 documents


In [11]:
import random

random.seed(42)

TARGET = 125000

with open("bn_news.txt", encoding="utf-8") as f:
    total_docs = sum(1 for _ in f)

print("Total news docs:", total_docs)

selected = set(
    random.sample(
        range(total_docs),
        TARGET
    )
)

count = 0

with open("bn_news.txt", encoding="utf-8") as inp, \
     open("news_125k.txt", "w", encoding="utf-8") as out:

    for idx, line in enumerate(inp):

        if idx in selected:
            out.write(line)
            count += 1

print("Saved:", count)

Total news docs: 250000
Saved: 125000


In [12]:
from collections import Counter

file_path = "news_125k.txt"

docs = 0
words = 0
chars = 0
vocab = Counter()

with open(file_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        docs += 1
        chars += len(line)

        ws = line.split()
        words += len(ws)

        vocab.update(ws)

print("Documents :", docs)
print("Characters:", chars)
print("Words     :", words)
print("Vocabulary:", len(vocab))
print("Avg Words/Doc:", round(words/docs, 2))

Documents : 125000
Characters: 37183477
Words     : 5657765
Vocabulary: 425932
Avg Words/Doc: 45.26


Merge

In [13]:
import random

# Load news
with open("news_125k.txt", encoding="utf-8") as f:
    news = [x.strip() for x in f if x.strip()]

# Load literature
with open("literature_balanced.txt", encoding="utf-8") as f:
    literature = [x.strip() for x in f if x.strip()]

print("News docs      :", len(news))
print("Literature docs:", len(literature))

# Merge
merged = news + literature

# Shuffle
random.seed(42)
random.shuffle(merged)

# Save
with open(
    "news_literature_merged.txt",
    "w",
    encoding="utf-8"
) as f:
    for doc in merged:
        f.write(doc + "\n")

print("Merged docs:", len(merged))

News docs      : 125000
Literature docs: 251038
Merged docs: 376038


In [14]:
from collections import Counter

chars = 0
words = 0
docs = 0
vocab = Counter()

with open(
    "news_literature_merged.txt",
    encoding="utf-8"
) as f:

    for line in f:
        line = line.strip()

        if not line:
            continue

        docs += 1
        chars += len(line)

        ws = line.split()

        words += len(ws)
        vocab.update(ws)

print("Documents :", docs)
print("Characters:", chars)
print("Words     :", words)
print("Vocabulary:", len(vocab))
print("TTR       :", len(vocab)/words)
print("Average Word Length:", chars/words)

Documents : 376038
Characters: 71121773
Words     : 11258936
Vocabulary: 827293
TTR       : 0.07347879053580196
Average Word Length: 6.3169177797973095


The merged News–Literature corpus contains 376,038 documents and approximately 11.26 million words. Although the number of documents differs between the two domains, their word counts are nearly identical, ensuring balanced lexical contributions during tokenizer training. The corpus exhibits a large vocabulary of 827,293 unique words, reflecting the complementary linguistic characteristics of formal news text and literary writing. The reduced Type–Token Ratio (0.0735) indicates increased lexical stability due to the larger corpus size, while the average word length remains consistent with observations from the individual corpora.


Transliteration

In [15]:
!pip install aksharamukha
from aksharamukha import transliterate
from tqdm import tqdm

INPUT_FILE = "news_literature_merged.txt"
OUTPUT_FILE = "news_literature_iso.txt"

with open(INPUT_FILE, encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    for line in tqdm(fin):
        line = line.strip()

        if not line:
            continue

        iso = transliterate.process(
            "Bengali",
            "ISO",
            line
        )

        fout.write(iso + "\n")

print("Done.")

376038it [08:51, 707.31it/s]

Done.


In [16]:
from collections import Counter

def corpus_stats(path):

    chars = 0
    words = 0
    vocab = Counter()

    with open(path, encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            chars += len(line)

            ws = line.split()

            words += len(ws)
            vocab.update(ws)

    return {
        "chars": chars,
        "words": words,
        "vocab": len(vocab),
        "avg_word_len": chars / words
    }

bn = corpus_stats("news_literature_merged.txt")
iso = corpus_stats("news_literature_iso.txt")

print("Bengali:", bn)
print("ISO:", iso)

print()
print("Expansion Factor:",
      iso["chars"] / bn["chars"])

print("Vocabulary Ratio:",
      iso["vocab"] / bn["vocab"])

Bengali: {'chars': 71121773, 'words': 11258936, 'vocab': 827293, 'avg_word_len': 6.3169177797973095}
ISO: {'chars': 81926995, 'words': 11258840, 'vocab': 819011, 'avg_word_len': 7.276681700779121}

Expansion Factor: 1.1519256557341448
Vocabulary Ratio: 0.9899890365324014


In [17]:
from aksharamukha import transliterate
import random

with open(
    "news_literature_merged.txt",
    encoding="utf-8"
) as f:

    docs = [
        x.strip()
        for x in f
        if x.strip()
    ]

random.seed(42)

sample = random.sample(
    docs,
    100
)

correct = 0

for txt in sample:

    iso = transliterate.process(
        "Bengali",
        "ISO",
        txt
    )

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso
    )

    if txt == back:
        correct += 1

print(
    f"Exact Match: {correct} / 100"
)

print(
    "Accuracy:",
    correct / 100
)

Exact Match: 12 / 100
Accuracy: 0.12


In [18]:
from sklearn.model_selection import train_test_split

with open(
    "news_literature_merged.txt",
    encoding="utf-8"
) as f:

    docs = [
        x.strip()
        for x in f
        if x.strip()
    ]

train_docs, test_docs = train_test_split(
    docs,
    test_size=0.10,
    random_state=42
)

with open(
    "train_nl_bn.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("\n".join(train_docs))

with open(
    "test_nl_bn.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("\n".join(test_docs))

print("Train:", len(train_docs))
print("Test :", len(test_docs))

Train: 338434
Test : 37604


In [19]:
from sklearn.model_selection import train_test_split

with open(
    "news_literature_iso.txt",
    encoding="utf-8"
) as f:

    docs = [
        x.strip()
        for x in f
        if x.strip()
    ]

train_docs, test_docs = train_test_split(
    docs,
    test_size=0.10,
    random_state=42
)

with open(
    "train_nl_iso.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("\n".join(train_docs))

with open(
    "test_nl_iso.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("\n".join(test_docs))

print("Train:", len(train_docs))
print("Test :", len(test_docs))

Train: 338433
Test : 37604


In [20]:
import os

for f in [
    "train_nl_bn.txt",
    "test_nl_bn.txt",
    "train_nl_iso.txt",
    "test_nl_iso.txt"
]:

    print(
        f,
        round(
            os.path.getsize(f)/(1024*1024),
            2
        ),
        "MB"
    )

train_nl_bn.txt 162.14 MB
test_nl_bn.txt 18.15 MB
train_nl_iso.txt 90.02 MB
test_nl_iso.txt 9.99 MB


In [21]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tok.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tok.train(
        ["train_nl_bn.txt"],
        trainer
    )

    tok.save(
        f"bpe_bn_{vocab_size}.json"
    )

    print(vocab_size)

5000
10000
15000
20000
25000
30000
35000
40000
45000
50000


In [22]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tok.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tok.train(
        ["train_nl_iso.txt"],
        trainer
    )

    tok.save(
        f"bpe_iso_{vocab_size}.json"
    )

    print(vocab_size)

5000
10000
15000
20000
25000
30000
35000
40000
45000
50000


In [23]:
from tokenizers import Tokenizer
import numpy as np

def evaluate_tokenizer(tokenizer_path, test_file):

    tok = Tokenizer.from_file(tokenizer_path)

    total_words = 0
    total_tokens = 0

    oov = 0

    token_counts = []

    with open(test_file, encoding="utf-8") as f:

        for line in f:

            ws = line.strip().split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                total_words += 1
                total_tokens += len(tks)

                token_counts.append(
                    len(tks)
                )

                if "[UNK]" in tks:
                    oov += 1

    fertility = total_tokens / total_words

    return {
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": 1 / fertility,
        "compression": 1 / fertility,
        "avg_tokens": fertility,
        "wfr": fertility - 1,
        "variance": np.var(token_counts)
    }

In [24]:
VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"bpe_bn_{vocab_size}.json",
        "test_nl_bn.txt"
    )

    r["vocab"] = vocab_size

    print(r)

print("\nFinished.")

{'oov': 3.000763429519569e-05, 'fertility': 1.8298505355038857, 'cpt': 0.5464927219996305, 'compression': 0.5464927219996305, 'avg_tokens': 1.8298505355038857, 'wfr': 0.8298505355038857, 'variance': np.float64(1.3768388679957), 'vocab': 5000}
{'oov': 3.000763429519569e-05, 'fertility': 1.5931017744220222, 'cpt': 0.6277062872287619, 'compression': 0.6277062872287619, 'avg_tokens': 1.5931017744220222, 'wfr': 0.5931017744220222, 'variance': np.float64(0.987932591793691), 'vocab': 10000}
{'oov': 3.000763429519569e-05, 'fertility': 1.4976624935461522, 'cpt': 0.6677071798948565, 'compression': 0.6677071798948565, 'avg_tokens': 1.4976624935461522, 'wfr': 0.4976624935461522, 'variance': np.float64(0.8167699068564416), 'vocab': 15000}
{'oov': 3.000763429519569e-05, 'fertility': 1.4436381608850486, 'cpt': 0.692694351739034, 'compression': 0.692694351739034, 'avg_tokens': 1.4436381608850486, 'wfr': 0.4436381608850486, 'variance': np.float64(0.7213534809060538), 'vocab': 20000}
{'oov': 3.000763429

In [25]:
for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"bpe_iso_{vocab_size}.json",
        "test_nl_iso.txt"
    )

    r["vocab"] = vocab_size

    print(r)

print("\nFinished.")

{'oov': 2.3141513917840505e-05, 'fertility': 1.823792502505514, 'cpt': 0.5483079893278466, 'compression': 0.5483079893278466, 'avg_tokens': 1.823792502505514, 'wfr': 0.823792502505514, 'variance': np.float64(1.2988483297154172), 'vocab': 5000}
{'oov': 2.3141513917840505e-05, 'fertility': 1.592677312949813, 'cpt': 0.6278735760653803, 'compression': 0.6278735760653803, 'avg_tokens': 1.592677312949813, 'wfr': 0.5926773129498131, 'variance': np.float64(0.940124425502216), 'vocab': 10000}
{'oov': 2.3141513917840505e-05, 'fertility': 1.4999127742936942, 'cpt': 0.6667054359016963, 'compression': 0.6667054359016963, 'avg_tokens': 1.4999127742936942, 'wfr': 0.4999127742936942, 'variance': np.float64(0.7866944229413231), 'vocab': 15000}
{'oov': 2.3141513917840505e-05, 'fertility': 1.4474705435229573, 'cpt': 0.6908603456386259, 'compression': 0.6908603456386259, 'avg_tokens': 1.4474705435229573, 'wfr': 0.44747054352295734, 'variance': np.float64(0.7010404037817127), 'vocab': 20000}
{'oov': 2.3141

### BPE Evaluation

BPE exhibited consistent improvements as vocabulary size increased from 5k to 50k. Fertility decreased steadily, indicating that larger vocabularies produced longer and more informative subword units. OOV rates remained effectively zero across all configurations, demonstrating strong vocabulary coverage.

For both Bengali and ISO representations, tokenization quality improved substantially between 5k and 30k vocabulary sizes, after which gains became progressively smaller. This suggests diminishing returns beyond approximately 30k–40k vocabulary.

Interestingly, the Bengali representation achieved slightly lower fertility and fragmentation than the ISO representation at larger vocabulary sizes, although the differences were marginal. The results indicate that BPE effectively captures lexical and morphological patterns present in both news and literary text while maintaining stable segmentation behavior.

In [26]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tok.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tok.train(
        ["train_nl_bn.txt"],
        trainer
    )

    tok.save(
        f"wp_bn_{vocab_size}.json"
    )

    print(vocab_size)

print("Finished.")

5000
10000
15000
20000
25000
30000
35000
40000
45000
50000
Finished.


In [27]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tok.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tok.train(
        ["train_nl_iso.txt"],
        trainer
    )

    tok.save(
        f"wp_iso_{vocab_size}.json"
    )

    print(vocab_size)

print("Finished.")

5000
10000
15000
20000
25000
30000
35000
40000
45000
50000
Finished.


In [28]:
VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_bn_{vocab_size}.json",
        "test_nl_bn.txt"
    )

    r["vocab"] = vocab_size

    print(r)

print("\nFinished.")

{'oov': 3.353794421227753e-05, 'fertility': 2.0021261291475625, 'cpt': 0.4994690321662033, 'compression': 0.4994690321662033, 'avg_tokens': 2.0021261291475625, 'wfr': 1.0021261291475625, 'variance': np.float64(1.7224557525560624), 'vocab': 5000}
{'oov': 3.353794421227753e-05, 'fertility': 1.6738726175924168, 'cpt': 0.5974170253399158, 'compression': 0.5974170253399158, 'avg_tokens': 1.6738726175924168, 'wfr': 0.6738726175924168, 'variance': np.float64(1.1774037112691), 'vocab': 10000}
{'oov': 3.353794421227753e-05, 'fertility': 1.552702672885896, 'cpt': 0.6440383065364166, 'compression': 0.6440383065364166, 'avg_tokens': 1.552702672885896, 'wfr': 0.5527026728858959, 'variance': np.float64(0.9583504064180641), 'vocab': 15000}
{'oov': 3.353794421227753e-05, 'fertility': 1.4860062927774271, 'cpt': 0.6729446603694694, 'compression': 0.6729446603694694, 'avg_tokens': 1.4860062927774271, 'wfr': 0.48600629277742713, 'variance': np.float64(0.831484515420952), 'vocab': 20000}
{'oov': 3.35379442

In [29]:
VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_iso_{vocab_size}.json",
        "test_nl_iso.txt"
    )

    r["vocab"] = vocab_size

    print(r)

print("\nFinished.")

{'oov': 2.5811688600668257e-05, 'fertility': 2.002257187665217, 'cpt': 0.499436339227767, 'compression': 0.499436339227767, 'avg_tokens': 2.002257187665217, 'wfr': 1.0022571876652169, 'variance': np.float64(1.6497071492788584), 'vocab': 5000}
{'oov': 2.5811688600668257e-05, 'fertility': 1.6799217104782995, 'cpt': 0.5952658351651903, 'compression': 0.5952658351651903, 'avg_tokens': 1.6799217104782995, 'wfr': 0.6799217104782995, 'variance': np.float64(1.1446327227358728), 'vocab': 10000}
{'oov': 2.5811688600668257e-05, 'fertility': 1.5601928578167583, 'cpt': 0.6409464028692843, 'compression': 0.6409464028692843, 'avg_tokens': 1.5601928578167583, 'wfr': 0.5601928578167583, 'variance': np.float64(0.937709966882329), 'vocab': 15000}
{'oov': 2.5811688600668257e-05, 'fertility': 1.49386838887, 'cpt': 0.6694030126418469, 'compression': 0.6694030126418469, 'avg_tokens': 1.49386838887, 'wfr': 0.49386838886999995, 'variance': np.float64(0.8162601705448809), 'vocab': 20000}
{'oov': 2.5811688600668

In [30]:
import pandas as pd

bn_results = []
iso_results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_bn_{vocab_size}.json",
        "test_nl_bn.txt"
    )
    r["vocab"] = vocab_size
    bn_results.append(r)

    r = evaluate_tokenizer(
        f"wp_iso_{vocab_size}.json",
        "test_nl_iso.txt"
    )
    r["vocab"] = vocab_size
    iso_results.append(r)

pd.DataFrame(bn_results).to_csv(
    "wp_bn_results.csv",
    index=False
)

pd.DataFrame(iso_results).to_csv(
    "wp_iso_results.csv",
    index=False
)

print("Saved.")

Saved.


### WordPiece Evaluation

WordPiece demonstrated the expected reduction in fertility and fragmentation as vocabulary size increased. OOV rates remained negligible across all experiments, indicating strong vocabulary coverage.

Compared with BPE, WordPiece consistently produced higher fertility and word fragmentation rates, suggesting a greater tendency to split words into smaller subword units. Although performance improved substantially between 5k and 30k vocabularies, gains became progressively smaller beyond 30k.

Across both Bengali and ISO representations, WordPiece exhibited slightly higher variance than BPE, indicating less stable segmentation behavior. Overall, WordPiece achieved competitive performance but remained consistently less efficient than BPE on the News–Literature corpus.

In [31]:
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(Unigram())

    tok.pre_tokenizer = Whitespace()

    trainer = UnigramTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"],
        unk_token="[UNK]"
    )

    tok.train(
        ["train_nl_bn.txt"],
        trainer
    )

    tok.save(
        f"uni_bn_{vocab_size}.json"
    )

    print(vocab_size)

print("Finished.")

5000
10000
15000
20000
25000
30000
35000
40000
45000
50000
Finished.


In [32]:
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(Unigram())

    tok.pre_tokenizer = Whitespace()

    trainer = UnigramTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"],
        unk_token="[UNK]"
    )

    tok.train(
        ["train_nl_iso.txt"],
        trainer
    )

    tok.save(
        f"uni_iso_{vocab_size}.json"
    )

    print(vocab_size)

print("Finished.")

5000
10000
15000
20000
25000
30000
35000
40000
45000
50000
Finished.


In [ ]:
from tokenizers import Tokenizer
import numpy as np

def evaluate_tokenizer(tokenizer_path, test_file):

    tok = Tokenizer.from_file(tokenizer_path)

    total_words = 0
    total_tokens = 0

    oov = 0

    token_counts = []

    with open(test_file, encoding="utf-8") as f:

        for line in f:

            ws = line.strip().split()

            for w in ws:

                try:
                    enc = tok.encode(w)
                    tks = enc.tokens

                except:
                    oov += 1
                    total_words += 1
                    token_counts.append(1)
                    continue

                total_words += 1
                total_tokens += len(tks)

                token_counts.append(
                    len(tks)
                )

                if "[UNK]" in tks:
                    oov += 1

    fertility = total_tokens / total_words

    return {
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": 1 / fertility,
        "compression": 1 / fertility,
        "avg_tokens": fertility,
        "wfr": fertility - 1,
        "variance": np.var(token_counts)
    }

In [34]:
VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"uni_bn_{vocab_size}.json",
        "test_nl_bn.txt"
    )

    r["vocab"] = vocab_size

    print(r)

print("\nFinished.")

{'oov': 0.0, 'fertility': 1.8267447453543328, 'cpt': 0.5474218565802035, 'compression': 0.5474218565802035, 'avg_tokens': 1.8267447453543328, 'wfr': 0.8267447453543328, 'variance': np.float64(1.4782104452881641), 'vocab': 5000}
{'oov': 0.0, 'fertility': 1.6026415543954564, 'cpt': 0.6239698435731732, 'compression': 0.6239698435731732, 'avg_tokens': 1.6026415543954564, 'wfr': 0.6026415543954564, 'variance': np.float64(1.0352848243694601), 'vocab': 10000}
{'oov': 0.0, 'fertility': 1.5157985781676808, 'cpt': 0.6597182596706315, 'compression': 0.6597182596706315, 'avg_tokens': 1.5157985781676808, 'wfr': 0.5157985781676808, 'variance': np.float64(0.8554015485276483), 'vocab': 15000}
{'oov': 0.0, 'fertility': 1.4690555097105586, 'cpt': 0.6807094717591886, 'compression': 0.6807094717591886, 'avg_tokens': 1.4690555097105586, 'wfr': 0.46905550971055865, 'variance': np.float64(0.7564044585640609), 'vocab': 20000}
{'oov': 0.0, 'fertility': 1.4403417339999736, 'cpt': 0.6942796812690414, 'compressio

In [35]:
VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"uni_iso_{vocab_size}.json",
        "test_nl_iso.txt"
    )

    r["vocab"] = vocab_size

    print(r)

print("\nFinished.")

{'oov': 0.0, 'fertility': 1.8630903533709176, 'cpt': 0.536742621306951, 'compression': 0.536742621306951, 'avg_tokens': 1.8630903533709176, 'wfr': 0.8630903533709176, 'variance': np.float64(1.5805221626687043), 'vocab': 5000}
{'oov': 0.0, 'fertility': 1.6331963237034968, 'cpt': 0.6122962594799153, 'compression': 0.6122962594799153, 'avg_tokens': 1.6331963237034968, 'wfr': 0.6331963237034968, 'variance': np.float64(1.1011549425414864), 'vocab': 10000}
{'oov': 0.0, 'fertility': 1.5471997878101185, 'cpt': 0.6463289407603809, 'compression': 0.6463289407603809, 'avg_tokens': 1.5471997878101185, 'wfr': 0.5471997878101185, 'variance': np.float64(0.9231519233734887), 'vocab': 15000}
{'oov': 0.0, 'fertility': 1.499909214060784, 'cpt': 0.6667070184152325, 'compression': 0.6667070184152325, 'avg_tokens': 1.499909214060784, 'wfr': 0.4999092140607839, 'variance': np.float64(0.8177343129372051), 'vocab': 20000}
{'oov': 0.0, 'fertility': 1.4700272891852586, 'cpt': 0.6802594804578326, 'compression': 0

In [36]:
import random

random.seed(42)

with open(
    "test_nl_bn.txt",
    encoding="utf-8"
) as f:

    sentences = [
        x.strip()
        for x in f
        if x.strip()
    ]

sample_sentences = random.sample(
    sentences,
    20
)

for i,s in enumerate(sample_sentences,1):
    print(f"{i}. {s}")

1. নতুন দেশ, শঙ্করের তরুণ তাজা মন— সে ইউগান্ডার এই নির্জন মাঠ ও বনে নিজের স্বপ্নের সার্থকতাকে যেন খুঁজে পেলে। কাজ শেষ হয়ে যেতেই সে তাঁবু থেকে রোজ বেরিয়ে পড়তো— যেদিকে দু’চোখ যায় সেদিকে বেড়াতে বের হত— পুবে, পশ্চিমে, দক্ষিণে, উত্তরে। সব দিকেই লম্বা লম্বা ঘাস। কোথাও মানুষের মাথা সমান উঁচু, কোথাও তার চেয়েও উঁচু।
2. ১৯৫১ সালে তারা নতুন একটি মসজিদ নির্মাণ করার জন্য সৌদি আরব সরকারসহ বিভিন্ন প্রতিষ্ঠান থেকে আর্থিক সহযোগিতা নিয়ে একটি তহবিল গঠন করে।]]. তারা তাইচুং মসজিদ নির্মাণের জন্য দক্ষিণ তাইচুং জেলার ঝংজিয়াও রোডের ১৬৫ নং লাইনের ১২ নং এর জাপানি শৈলীতে তৈরী ভবনটিকে পছন্দ করে।. ঐ ভবনটির আয়তন ছিল ১৩০ বর্গ মিটার।. ১৯৭৫ সালের এপ্রিলে সৌদি আরবের যোগাযোগ মন্ত্রনালয়ের কর্মকর্তা মসজিদটি দেখতে যান এবং তিনি মসজিদের মেরামতের দুর্দশা দেখে নতুন একটি জায়গায় তাইচুং মসজিদ পুনরায় নির্মাণের জন্য চীনা মুসলিম এসোসিয়েশনকে একটি তহবিল গঠন করে দেন।
3. আর্থিক সহায়তা পাওয়ার জন্য মূল দরখাস্ত জমা দেওয়া
4. গোল্ডফিংগারের ক্ষমতাকে নিচু নজরে দেখার কোন ইচ্ছে বন্ডের নেই। কিন্তু তাই করে ফেলেছে, পিস্তলটাও সঙ্গে আ

In [37]:
from tokenizers import Tokenizer

bpe = Tokenizer.from_file(
    "bpe_bn_50000.json"
)

wp = Tokenizer.from_file(
    "wp_bn_50000.json"
)

uni = Tokenizer.from_file(
    "uni_bn_50000.json"
)

In [38]:
for i,sent in enumerate(sample_sentences,1):

    print("="*80)
    print(f"Sentence {i}")
    print(sent)

    print("\nBPE")
    print(bpe.encode(sent).tokens)

    print("\nWordPiece")
    print(wp.encode(sent).tokens)

    print("\nUnigram")
    print(uni.encode(sent).tokens)

Sentence 1
নতুন দেশ, শঙ্করের তরুণ তাজা মন— সে ইউগান্ডার এই নির্জন মাঠ ও বনে নিজের স্বপ্নের সার্থকতাকে যেন খুঁজে পেলে। কাজ শেষ হয়ে যেতেই সে তাঁবু থেকে রোজ বেরিয়ে পড়তো— যেদিকে দু’চোখ যায় সেদিকে বেড়াতে বের হত— পুবে, পশ্চিমে, দক্ষিণে, উত্তরে। সব দিকেই লম্বা লম্বা ঘাস। কোথাও মানুষের মাথা সমান উঁচু, কোথাও তার চেয়েও উঁচু।

BPE
['নতুন', 'দেশ', ',', 'শঙ্করের', 'তরুণ', 'তাজা', 'মন', '—', 'সে', 'ইউ', 'গান', '্ডার', 'এই', 'নির্জন', 'মাঠ', 'ও', 'বনে', 'নিজের', 'স্বপ্নের', 'সার্থক', 'তাকে', 'যেন', 'খুঁজে', 'পেলে', '।', 'কাজ', 'শেষ', 'হয়ে', 'যেতেই', 'সে', 'তাঁবু', 'থেকে', 'রোজ', 'বেরিয়ে', 'পড়তো', '—', 'যেদিকে', 'দু', '’', 'চোখ', 'যায়', 'সেদিকে', 'বেড়াতে', 'বের', 'হত', '—', 'পুবে', ',', 'পশ্চিমে', ',', 'দক্ষিণে', ',', 'উত্তরে', '।', 'সব', 'দিকেই', 'লম্বা', 'লম্বা', 'ঘাস', '।', 'কোথাও', 'মানুষের', 'মাথা', 'সমান', 'উঁচু', ',', 'কোথাও', 'তার', 'চেয়েও', 'উঁচু', '।']

WordPiece
['নতুন', 'দেশ', ',', 'শঙ্করের', 'তরুণ', 'তাজা', 'মন', '—', 'সে', 'ইউ', '##গান', '##্ডার', 'এই', 'নির্জন', 'মাঠ', 'ও', 

In [39]:
import pandas as pd

rows = []

for sent in sample_sentences:

    rows.append({
        "Sentence": sent,
        "BPE": " ".join(
            bpe.encode(sent).tokens
        ),
        "WordPiece": " ".join(
            wp.encode(sent).tokens
        ),
        "Unigram": " ".join(
            uni.encode(sent).tokens
        )
    })

df = pd.DataFrame(rows)

df.to_csv(
    "morphological_analysis.csv",
    index=False
)

df.head()

,Sentence,BPE,WordPiece,Unigram
0,"নতুন দেশ, শঙ্করের তরুণ তাজা মন— সে ইউগান্ডার এ...","নতুন দেশ , শঙ্করের তরুণ তাজা মন — সে ইউ গান ্ড...","নতুন দেশ , শঙ্করের তরুণ তাজা মন — সে ইউ ##গান ...","নতুন দেশ , শঙ্করের তরুণ তাজা মন — সে ই উগান্ডা..."
1,১৯৫১ সালে তারা নতুন একটি মসজিদ নির্মাণ করার জন...,১৯৫১ সালে তারা নতুন একটি মসজিদ নির্মাণ করার জন...,১৯৫১ সালে তারা নতুন একটি মসজিদ নির্মাণ করার জন...,১৯ ৫১ সালে তারা নতুন একটি মসজিদ নির্মাণ করার জ...
2,আর্থিক সহায়তা পাওয়ার জন্য মূল দরখাস্ত জমা দে...,আর্থিক সহায়তা পাওয়ার জন্য মূল দরখাস্ত জমা দে...,আর্থিক সহায়তা পাওয়ার জন্য মূল দরখাস্ত জমা দে...,আর্থিক সহায়তা পাওয়ার জন্য মূল দরখাস্ত জমা দে...
3,গোল্ডফিংগারের ক্ষমতাকে নিচু নজরে দেখার কোন ইচ্...,গোল্ডফিং গারের ক্ষমতাকে নিচু নজরে দেখার কোন ইচ...,গোল্ডফিং ##গারের ক্ষমতাকে নিচু নজরে দেখার কোন ...,গোল্ডফিংগার ের ক্ষমতা কে নিচু নজরে দেখার কোন ই...
4,৫. নিয়মিত জুতো রোদে দিন।,৫ . নিয়মিত জুতো রোদে দিন ।,৫ . নিয়মিত জুতো রোদে দিন ।,৫ . নিয়মিত জুতো রোদে দিন ।


In [40]:
def fertility_on_sentences(tok, sentences):

    words = 0
    pieces = 0

    for s in sentences:

        for w in s.split():

            pieces += len(
                tok.encode(w).tokens
            )

            words += 1

    return pieces / words

print(
    "BPE:",
    fertility_on_sentences(
        bpe,
        sample_sentences
    )
)

print(
    "WordPiece:",
    fertility_on_sentences(
        wp,
        sample_sentences
    )
)

print(
    "Unigram:",
    fertility_on_sentences(
        uni,
        sample_sentences
    )
)

BPE: 1.263013698630137
WordPiece: 1.2808219178082192
Unigram: 1.3424657534246576


In [42]:
import os

for f in os.listdir():
    print(f)

.config
wp_bn_15000.json
bpe_iso_15000.json
bpe_bn_20000.json
uni_bn_50000.json
uni_bn_45000.json
uni_bn_30000.json
news_literature_iso.txt
bpe_iso_50000.json
bpe_iso_5000.json
wp_iso_5000.json
news_literature_merged.txt
uni_iso_5000.json
wp_iso_15000.json
uni_iso_25000.json
bpe_iso_35000.json
uni_iso_45000.json
wp_bn_25000.json
bn_news.txt
bpe_bn_35000.json
bpe_bn_25000.json
wp_bn_5000.json
uni_iso_50000.json
literature_balanced.txt
bpe_bn_5000.json
wp_bn_30000.json
bn_literature.txt
wp_bn_40000.json
wp_iso_20000.json
bpe_bn_40000.json
uni_iso_20000.json
test_nl_iso.txt
bpe_bn_10000.json
bpe_bn_15000.json
bpe_iso_45000.json
wp_bn_results.csv
news_125k.txt
uni_bn_20000.json
uni_bn_10000.json
morphological_analysis.csv
bpe_bn_50000.json
uni_iso_15000.json
wp_bn_20000.json
uni_bn_40000.json
wp_iso_45000.json
train_nl_bn.txt
wp_bn_35000.json
bpe_bn_30000.json
wp_iso_10000.json
wp_iso_25000.json
wp_iso_30000.json
uni_iso_35000.json
train_nl_iso.txt
uni_iso_40000.json
bpe_iso_40000.json
bpe